In [1]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
import re

dataset = load_dataset("gsm8k", "main", split="train")
dataset[0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

In [2]:
# Function to clean answer
def clean_answer(answer):
    return re.sub(r"<<.*?>>", "", answer).strip()

# Transform the dataset in-place
def transform(example):
    prompt = example["question"].strip()
    completion = re.sub(r"<<.*?>>", "", example["answer"]).strip()
    return {
        "prompt": prompt,
        "completion": completion
    }

# Apply transformation
transformed_dataset = dataset.map(transform, remove_columns=dataset.column_names)

# Check an example
print(transformed_dataset[0])

{'prompt': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'completion': 'Natalia sold 48/2 = 24 clips in May.\nNatalia sold 48+24 = 72 clips altogether in April and May.\n#### 72'}


In [6]:
from transformers import AutoTokenizer

In [10]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
count = 0
for sample in transformed_dataset:
    # sample = transformed_dataset[100]
    prompt = sample["prompt"]
    completion = sample["completion"]

    # Tokenize parts
    tokens_prompt = tokenizer(prompt, add_special_tokens=False).input_ids
    tokens_full = tokenizer(prompt + completion, add_special_tokens=False).input_ids

    # Check alignment
    if tokens_prompt != tokens_full[:len(tokens_prompt)]:
        print("Token mismatch detected")
        print(tokens_prompt)
        print(tokens_full[:len(tokens_prompt)])
        print(f"Prompt: {prompt}")
        print(f"Completion: {completion}")
        count += 1
        if count > 2:
            break
    else:
        # print("Token alignment is fine")
        pass

Token mismatch detected
[644, 264, 11092, 11, 1070, 527, 220, 1627, 18718, 2653, 45526, 11, 220, 868, 6307, 2653, 45526, 11, 323, 220, 1187, 14071, 2653, 45526, 13, 220, 1442, 22770, 5097, 3201, 220, 19, 18718, 2653, 45526, 11, 323, 3842, 5097, 3201, 220, 21, 18718, 2653, 45526, 323, 11157, 439, 1690, 6307, 2653, 45526, 439, 279, 1396, 315, 18718, 2653, 45526, 430, 568, 7108, 11, 1243, 11294, 279, 2860, 1396, 315, 2653, 45526, 430, 14958, 304, 279, 11092, 13]
[644, 264, 11092, 11, 1070, 527, 220, 1627, 18718, 2653, 45526, 11, 220, 868, 6307, 2653, 45526, 11, 323, 220, 1187, 14071, 2653, 45526, 13, 220, 1442, 22770, 5097, 3201, 220, 19, 18718, 2653, 45526, 11, 323, 3842, 5097, 3201, 220, 21, 18718, 2653, 45526, 323, 11157, 439, 1690, 6307, 2653, 45526, 439, 279, 1396, 315, 18718, 2653, 45526, 430, 568, 7108, 11, 1243, 11294, 279, 2860, 1396, 315, 2653, 45526, 430, 14958, 304, 279, 11092, 34001]
Prompt: In a truck, there are 26 pink hard hats, 15 green hard hats, and 24 yellow hard hats.

In [3]:
training_args = SFTConfig(
    max_length=256,
    output_dir="/tmp",
)
trainer = SFTTrainer(
    "meta-llama/Llama-3.2-1B-Instruct",
    train_dataset=transformed_dataset,
    args=training_args,
)
trainer.train()

Truncating train dataset:   0%|          | 0/7473 [00:00<?, ? examples/s]

Step,Training Loss
500,0.792900
1000,0.683100
1500,0.376000
2000,0.298100
2500,0.104100


TrainOutput(global_step=2805, training_loss=0.4120700482591163, metrics={'train_runtime': 10055.3174, 'train_samples_per_second': 2.23, 'train_steps_per_second': 0.279, 'total_flos': 2.7853088288661504e+16, 'train_loss': 0.4120700482591163})

In [6]:
trainer.save_model("./tmp/fine_tuned_llama")